In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
user_gameplay_df = pd.read_csv('/content/UserGameplay data.csv', skiprows=3)
deposit_df = pd.read_csv('/content/Deposit data.csv', skiprows=3)
withdrawal_df = pd.read_csv('/content/Withdrawal data.csv', skiprows=3)

Need to standardize the column headers

In [ ]:
user_gameplay_df.columns = ['User ID', 'Games Played', 'Datetime']
deposit_df.columns = ['User ID', 'Datetime', 'Amount']
withdrawal_df.columns = ['User ID', 'Datetime', 'Amount']

**Part A: Calculate Loyalty Points**

Parsing the dates and standardizing

In [ ]:
def date_parser(df, col):
    df[col] = pd.to_datetime(df[col], dayfirst=True)
    return df

In [ ]:
user_gameplay_df = date_parser(user_gameplay_df, 'Datetime')
deposit_df = date_parser(deposit_df, 'Datetime')
withdrawal_df = date_parser(withdrawal_df, 'Datetime')

Dates seem parsed fine

In [ ]:
class slot_and_date_creator:

  def __init__(self, datetime_col='Datetime'):
    self.datetime_col = datetime_col

  @staticmethod

  def get_time_slots(dt):
    if 0 <= dt.hour < 12:
        return 'S1'
    else:
        return 'S2'

  def create_slots_column(self,dictframes):
    for name,df in dictframes.items():
        df[self.datetime_col] = pd.to_datetime(df[self.datetime_col], dayfirst=True)
        df['Slot'] = df[self.datetime_col].apply(self.get_time_slots)
        df['Date'] = df[self.datetime_col].dt.date
    return dictframes


In [ ]:
slot_create = slot_and_date_creator()

data = {
    'gameplay': user_gameplay_df,
    'deposit': deposit_df,
    'withdrawal': withdrawal_df
}

slots_data = slot_create.create_slots_column(data)

In [ ]:
user_gameplay_new = slots_data['gameplay']
deposit_new = slots_data['deposit']
withdrawal_new = slots_data['withdrawal']

In [ ]:
#print(deposit_new[['Datetime', 'Slot', 'Date']].head())

In [ ]:
class loyalty_points_calculator:

    def __init__(self, slot_data):

        self.data = slot_data

        self.config = {
            'deposit': ('Amount', ['sum', 'count'], ['dep_amt', 'dep_count']),
            'withdrawal': ('Amount', ['sum', 'count'], ['with_amt', 'with_count']),
            'gameplay': ('Games Played', ['sum'], ['games_played'])
        }


    def data_aggregate(self, req_dict):

      aggregate_list = []

      for key, (col, funcs, new_names) in self.config.items():
          df = req_dict[key]
          agg = df.groupby('User ID')[col].agg(funcs)
          agg.columns = new_names
          aggregate_list.append(agg)

      master = pd.concat(aggregate_list, axis=1).fillna(0)

      master['Total_Points'] = (
            (master['dep_amt'] * 0.01) +
            (master['with_amt'] * 0.005) +
            (master['games_played'] * 0.2) +
            ((master['dep_count'] - master['with_count']).clip(lower=0) * 0.001)
        )

      return master


    def calculate_loyalty_points(self, req_date_str, req_slot):

      req_date_obj = pd.to_datetime(req_date_str).date()

      loyal_set={}

      for key in self.data:
          df = self.data[key]
          loyal_set[key] = df[(df['Date'] == req_date_obj) & (df['Slot'] == req_slot)]

      result = self.data_aggregate(loyal_set)

      return result[['Total_Points']].sort_values(by='Total_Points', ascending=False)


    def get_leaderboard(self, month_num, year_num):
      req_set = {}
      for key in self.data:
          df = self.data[key]
          req_month_yr = (df['Datetime'].dt.month == month_num) & (df['Datetime'].dt.year == year_num)
          req_set[key] = df[req_month_yr]

      loyaltypoints_data = self.data_aggregate(req_set)

      leaderboard = loyaltypoints_data.sort_values(by=['Total_Points', 'games_played'], ascending=False)
      leaderboard['Rank'] = range(1, len(leaderboard) + 1)

      return leaderboard[['Total_Points', 'games_played', 'Rank']]

In [ ]:
calculator = loyalty_points_calculator(slots_data)
result1 = calculator.calculate_loyalty_points('2022-10-02', 'S1')
#result = calculator.calculate_loyalty_points('2022-10-16', 'S2')
#result = calculator.calculate_loyalty_points('2022-10-18', 'S1')
#result = calculator.calculate_loyalty_points('2022-10-26', 'S2')

#print(result)

In [ ]:
result_leader = calculator.get_leaderboard(10,2022)
#print(result_leader)

Top 50 players

In [ ]:
#print(result_leader.head(50))

**Part-B: Bonus money allocation**

A simple way to allocate the bonus money could be to allocate in proportion to the player's points divided by the games played (more points earned playing less games would get a larger share)

In [ ]:
def get_bonus_allocation_amt(leaderboard, tot_bonus):
  leaderboard['performance_ratio'] = (
        leaderboard['Total_Points'] / leaderboard['games_played'].replace(0, np.nan)
    ).fillna(0)

  if leaderboard['performance_ratio'].sum() > 0:
      leaderboard['allocated_bonus'] = (
            leaderboard['performance_ratio'] / leaderboard['performance_ratio'].sum()
        ) * tot_bonus
  else:
      leaderboard['allocated_bonus'] = 0

  return leaderboard.sort_values(by='allocated_bonus', ascending=False)


In [ ]:
#print(get_bonus_allocation_amt(result_leader.head(50),50000))

There could be other ideas where a maximum limit on the bonus that can be allocated to a user irrespective of performance ratio could be implemented, leading to a balancing out of the total allocation amount preventing excessively unequal allocations. This could contribute towards retention

**Part C: Improving the loyalty points calculation formula**

***The formula: (0.01 * deposit) + (0.005 * withdrawal) + 0.001 * {max(deposit-withdrawal),0} + 0.02 * no. of games played***

The formula is weighted linear sum.

1. The first problem it is one size fits all. Most gaming platforms have subscription/deposit tiers in terms of how much money and time a player is willing to invest and what are the benefits. The weights may be adjusted based on the deposit amount and number of games played (based on each tier) to factor in a reward mechanism for larger deposits and greater game play times.

2. The formula takes care of the negative problem for deposit - withdrawal setting it to 0 but does not account for deliberate deposit and withdrawals to gain points. The deposit weight needs to be placed under conditioned adjustment as how must withdrawal is tolerable against a certain deposit.

3. Cases with very low game-plays versus points accrued (if points are to be obtained at all and if yes, then how much) needs to be factored in to maintain a competitive and dynamic game play environment.